# Social Network Analysis: Repeated Characters in the Bible and Q'uran

In [6]:
# import packages
import pandas as pd
import kagglehub
import os
import itertools
import networkx as nx
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
import re
import itertools
import spacy

## Load Textual Data

### BibleData

In [ ]:
# latest version of BibleData
path = kagglehub.dataset_download("bradystephenson/bibledata")

person_verse_path = os.path.join(path, "BibleData-PersonVerse.csv")
person_names_path = os.path.join(path, "BibleData-Person.csv")
person_relationship_path = os.path.join(path, "BibleData-PersonVerseApostolic.csv")

df_relation_pv = pd.read_csv(person_verse_path)
df_relation_names = pd.read_csv(person_names_path)
df_relation = pd.read_csv(person_relationship_path)

df_relation.head()

,person_verse_id,reference_id,person_label_id,person_id,person_label,person_label_count,person_verse_sequence,person_verse_notes
0,MAT 1:1__YHVH_1_123,MAT 1:1,YHVH_1_123,YHVH_1,Jesus,1.0,34517,NaN
1,MAT 1:1__YHVH_1_124,MAT 1:1,YHVH_1_124,YHVH_1,The Messiah,1.0,34518,NaN
2,MAT 1:1__YHVH_1_125,MAT 1:1,YHVH_1_125,YHVH_1,Son of David,1.0,34519,NaN
3,MAT 1:1__YHVH_1_126,MAT 1:1,YHVH_1_126,YHVH_1,Son of Abraham,1.0,34520,NaN
4,MAT 1:2__Abram_1_2,MAT 1:2,Abram_1_2,Abram_1,Abraham,1.0,34521,NaN


In [10]:
# gospel book matching
gospel_books = ["MAT", "MRK", "LUK", "JHN"]
# extract book code
df_relation['book'] = df_relation['reference_id'].str[:3]
# filter to four gospels only
gospels_df = df_relation[df_relation['book'].isin(gospel_books)].copy()
gospels_df = gospels_df.dropna(subset=['person_id'])

print(f"Total entries: {len(gospels_df)}")
print(f"Unique characters: {gospels_df['person_id'].nunique()}")

Total entries: 2506
Unique characters: 198


In [12]:
verse_groups = gospels_df.groupby('reference_id')['person_id'].apply(set).reset_index()

multi_person_verses = verse_groups[verse_groups['person_id'].apply(len) > 1]
print(f"Verses with co-occurring characters: {len(multi_person_verses)}")

Verses with co-occurring characters: 392


### Qur'an Data

In [14]:
# !python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 34.9 MB/s  0:00:00eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [16]:
# Qur'an API
nlp = spacy.load("en_core_web_sm")

url = "https://api.alquran.cloud/v1/quran/en.sahih"
response = requests.get(url).json()

verses_data = []
for surah in response['data']['surahs']:
    surah_num = surah['number']
    for ayah in surah['ayahs']:
        verses_data.append({
            'reference_id': f"QURAN {surah_num}:{ayah['numberInSurah']}",
            'text': ayah['text']
        })

df_quran = pd.DataFrame(verses_data)
df_quran.head()

,reference_id,text
0,QURAN 1:1,"In the name of Allah, the Entirely Merciful, t..."
1,QURAN 1:2,"[All] praise is [due] to Allah, Lord of the wo..."
2,QURAN 1:3,"The Entirely Merciful, the Especially Merciful,"
3,QURAN 1:4,Sovereign of the Day of Recompense.
4,QURAN 1:5,It is You we worship and You we ask for help.


In [18]:
def extract_characters(text):
    doc = nlp(text)
    discovered_names = set()

    for ent in doc.ents:
        if ent.label_ == "PERSON":
            clean_name = ent.text.strip().strip("'s").title()
            if len(clean_name) > 1:
                discovered_names.add(clean_name)
                
    return list(discovered_names)

In [20]:
df_quran['characters'] = df_quran['text'].apply(extract_characters)

df_quran.head()

,reference_id,text,characters
0,QURAN 1:1,"In the name of Allah, the Entirely Merciful, t...",[]
1,QURAN 1:2,"[All] praise is [due] to Allah, Lord of the wo...",[]
2,QURAN 1:3,"The Entirely Merciful, the Especially Merciful,",[]
3,QURAN 1:4,Sovereign of the Day of Recompense.,[]
4,QURAN 1:5,It is You we worship and You we ask for help.,[]


In [23]:
all_unique_names = sorted(list(set(
    name for sublist in df_quran['characters'] for name in sublist
)))

print(all_unique_names)
print("characters found:", len(all_unique_names))

["A'Ibah", 'Aaron', 'Abraham', 'Abu Lahab', 'Adam', 'Ahmad', 'Al-Ahqaf', 'Al-Firdau', 'Al-Kawthar', 'Al-Khidh', 'Al-Lat', 'Al-Madinah', 'Al-Marwah', 'Al-Masjid Al-Aqsa', 'Al-Masjid Al-Haram', 'Angel', 'Ansar', 'Arafat', 'Arise', 'Ascendant', 'Ayn', 'Bestower', 'Blessed', 'Comfort Wherein', 'Creator', 'David', 'Dhul-Kifl', 'Dhul-Qarnayn', 'Disposer', 'Dwell', 'Elia', 'Elisha', 'Ezra', 'Fire', 'Flame', 'Gabriel', 'Gog', 'Goliath', 'Gospel', 'Hajj', 'Haman', 'Haram', 'Harut', 'Hellfire', 'Him', 'Him Isaac', 'His Messenger', 'His Messenger - Allah', 'His Messenger - He', 'His Servant', 'Hud', 'Iblee', 'Isaac', 'Ishmael', 'Jacob', 'Jesu', 'Jinn', 'Job', 'John', 'Jonah', 'Joseph', 'Kaf', 'Kill Joseph', 'Kind', 'Lam', 'Lot', 'Luqman', 'Madinah', 'Madyan', 'Magog', 'Makkah', 'Mary', 'Mary - Distinguished', 'Mary - The', 'Masjid Al-Haram', 'Messenger', 'Messiah', 'Michael', 'Muhammad', 'Nasr', 'Noah', 'O Haman', 'Our Messenger', 'Pharaoh', 'Powerful', 'Praiseworthy', 'Prevailing', 'Prophet', 'P

In [24]:
QURAN_CHARACTER_MAP = {
    # Shared Characters
    "Jesus": "Jesus_1", "Jesu": "Jesus_1", "Said Jesu": "Jesus_1", "Messiah": "Jesus_1",
    "Mary": "Mary_1", "Mary - Distinguished": "Mary_1", "Mary - The": "Mary_1",
    "Zachariah": "Zachariah_1", "Zechariah": "Zachariah_1",
    "John": "John_Baptist_1",  # Assuming John is the Baptist in these contexts
    "Gabriel": "Gabriel_1",
    "Michael": "Michael_1",
    "Satan": "Satan_1", "Iblee": "Satan_1", # Iblis is the Quranic equivalent of Satan/Lucifer
    
    # Shared Ancestors
    "Abraham": "Abraham_1",
    "Moses": "Moses_1", "Said Mose": "Moses_1",
    "Aaron": "Aaron_1",
    "Adam": "Adam_1",
    "David": "David_1",
    "Elisha": "Elisha_1",
    "Ezra": "Ezra_1",
    "Isaac": "Isaac_1", "Him Isaac": "Isaac_1",
    "Ishmael": "Ishmael_1",
    "Jacob": "Jacob_1",
    "Job": "Job_1",
    "Jonah": "Jonah_1",
    "Joseph": "Joseph_1", "Kill Joseph": "Joseph_1",
    "Lot": "Lot_1",
    "Noah": "Noah_1",
    "Solomon": "Solomon_1",
    "Goliath": "Goliath_1",
    "Saul": "Saul_1",
    
    # Quran Specific Characters
    "Muhammad": "Muhammad_1", "Prophet Muhammad": "Muhammad_1",
    "Abu Lahab": "Abu_Lahab_1",
    "Dhul-Kifl": "Dhul_Kifl_1",
    "Dhul-Qarnayn": "Dhul_Qarnayn_1",
    "Haman": "Haman_1", "O Haman": "Haman_1",
    "Harut": "Harut_1",
    "Hud": "Hud_1",
    "Luqman": "Luqman_1",
    "Pharaoh": "Pharaoh_1", "Said Pharaoh": "Pharaoh_1",
    "Salih": "Salih_1",
    "Zayd": "Zayd_1"
}

In [26]:
def clean_and_resolve(raw_names):
    cleaned_ids = set()
    for name in raw_names:
        if name in QURAN_CHARACTER_MAP:
            cleaned_ids.add(QURAN_CHARACTER_MAP[name])
    return list(cleaned_ids)
# resolution mapping
df_quran['cleaned_characters'] = df_quran['characters'].apply(clean_and_resolve)
# filter to verses w/ characters
df_quran_clean = df_quran[df_quran['cleaned_characters'].map(len) > 0].copy()

df_quran_clean.head()

,reference_id,text,characters,cleaned_characters
29,QURAN 2:23,And if you are in doubt about what We have sen...,[Muhammad],[Muhammad_1]
36,QURAN 2:30,"And [mention, O Muhammad], when your Lord said...",[Muhammad],[Muhammad_1]
37,QURAN 2:31,And He taught Adam the names - all of them. Th...,[Adam],[Adam_1]
43,QURAN 2:37,"Then Adam received from his Lord [some] words,...",[Adam],[Adam_1]
68,QURAN 2:62,"Indeed, those who believed and those who were ...","[Prophet Muhammad, Sabean]",[Muhammad_1]


### Shared Character Mapping

### Network Graph Visualizations

### Comparative Analysis